# Chapter 13 — Serving and Systems Tradeoffs: Theory Notebook

Interactive exploration of the mathematics behind LLM serving:
bandwidth-bound decoding, queuing theory, batching strategies,
PagedAttention, speculative decoding, cost modelling, and capacity planning.

In [ ]:
# === Setup ===
import numpy as np
np.random.seed(42)

try:
    import matplotlib
    matplotlib.use('Agg')
    import matplotlib.pyplot as plt
    HAS_MPL = True
except ImportError:
    HAS_MPL = False

def print_table(headers, rows, col_width=16):
    """Print a formatted table."""
    hdr = " | ".join(h.center(col_width) for h in headers)
    print(hdr)
    print("-" * len(hdr))
    for row in rows:
        print(" | ".join(str(v).center(col_width) for v in row))

def fmt_si(val, unit=""):
    """Format with SI prefix."""
    for suffix, div in [("T", 1e12), ("G", 1e9), ("M", 1e6), ("K", 1e3)]:
        if abs(val) >= div:
            return f"{val/div:.2f} {suffix}{unit}"
    return f"{val:.2f} {unit}"

def fmt_bytes(val):
    """Format bytes with appropriate unit."""
    for suffix, div in [("TB", 1e12), ("GB", 1e9), ("MB", 1e6), ("KB", 1e3)]:
        if abs(val) >= div:
            return f"{val/div:.2f} {suffix}"
    return f"{val:.0f} B"

print("Setup complete ✓")

## §1 Bandwidth-Bound Decode Speed

During autoregressive decoding at small batch sizes, the GPU is
**memory-bandwidth-bound**: it spends most of its time reading model
weights from HBM, not computing.

**Decode throughput** (tokens/s) at batch size $B$:

$$\Theta(B) = \frac{B \times \text{BW}_{\text{HBM}}}{P \times b}$$

where $P$ = parameters, $b$ = bytes per parameter, BW = HBM bandwidth.

**Per-user TPOT** (time per output token, ms):

$$\text{TPOT}(B) = \frac{1000}{\Theta(B) / B} = \frac{1000 \times P \times b}{\text{BW}_{\text{HBM}}}$$

Note: TPOT is *constant* in the bandwidth-bound regime — independent of $B$.

In [ ]:
# === §1: Bandwidth-Bound Decode Speed ===

# --- 1a: Decode throughput vs batch size ---
# H100 SXM specs
HBM_BW = 3.35e12       # bytes/s (3.35 TB/s)
PEAK_FLOPS = 989e12    # BF16 FLOP/s

# LLaMA-3 8B in BF16
P_8B = 8e9
b_bf16 = 2  # bytes per param

batch_sizes = np.array([1, 2, 4, 8, 16, 32, 64, 128, 256, 512])

# Bandwidth-bound throughput
throughput_bw = batch_sizes * HBM_BW / (P_8B * b_bf16)

# Compute-bound throughput: 2 FLOPs per param per token
throughput_compute = PEAK_FLOPS / (2 * P_8B) * np.ones_like(batch_sizes, dtype=float)

# Actual throughput = min(BW-bound, compute-bound)
throughput_actual = np.minimum(throughput_bw, throughput_compute)

# TPOT (per-user, ms)
tpot = 1000.0 / (throughput_actual / batch_sizes)

# Critical batch size (BW→compute transition)
B_star = PEAK_FLOPS * b_bf16 / (2 * HBM_BW)

print("=" * 75)
print("BANDWIDTH-BOUND DECODE ANALYSIS — LLaMA-3 8B, BF16, H100")
print("=" * 75)
print(f"\nModel params:       {fmt_si(P_8B, 'params')}")
print(f"Bytes per param:    {b_bf16}")
print(f"Model size in HBM:  {fmt_bytes(P_8B * b_bf16)}")
print(f"HBM bandwidth:      {fmt_si(HBM_BW, 'B/s')}")
print(f"Peak BF16 FLOP/s:   {fmt_si(PEAK_FLOPS, 'FLOP/s')}")
print(f"Critical batch B*:  {B_star:.0f}")
print()

headers = ["Batch B", "Throughput", "Per-user TPOT", "Regime"]
rows = []
for i, B in enumerate(batch_sizes):
    regime = "BW-bound" if B < B_star else "Compute-bound"
    rows.append([
        str(int(B)),
        f"{throughput_actual[i]:,.0f} tok/s",
        f"{tpot[i]:.2f} ms",
        regime
    ])
print_table(headers, rows, col_width=18)

print(f"\n► Below B* = {B_star:.0f}, TPOT is constant at {tpot[0]:.2f} ms.")
print(f"► Above B*, throughput saturates but per-user TPOT grows linearly.")

In [ ]:
# --- 1b: Throughput and TPOT visualisation ---
if HAS_MPL:
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

    # Throughput vs batch size
    ax1.loglog(batch_sizes, throughput_bw, 'b--', label='BW-bound', alpha=0.7)
    ax1.loglog(batch_sizes, throughput_compute, 'r--', label='Compute-bound', alpha=0.7)
    ax1.loglog(batch_sizes, throughput_actual, 'k-o', label='Actual', linewidth=2)
    ax1.axvline(B_star, color='green', linestyle=':', label=f'B* = {B_star:.0f}')
    ax1.set_xlabel('Batch Size')
    ax1.set_ylabel('Throughput (tok/s)')
    ax1.set_title('Decode Throughput vs Batch Size')
    ax1.legend()
    ax1.grid(True, alpha=0.3)

    # TPOT vs batch size
    ax2.semilogx(batch_sizes, tpot, 'k-o', linewidth=2)
    ax2.axvline(B_star, color='green', linestyle=':', label=f'B* = {B_star:.0f}')
    ax2.set_xlabel('Batch Size')
    ax2.set_ylabel('Per-user TPOT (ms)')
    ax2.set_title('Per-User Latency vs Batch Size')
    ax2.legend()
    ax2.grid(True, alpha=0.3)

    plt.tight_layout()
    plt.savefig('throughput_vs_batch.png', dpi=100, bbox_inches='tight')
    plt.show()
    print("Plot saved: throughput_vs_batch.png")
else:
    print("(Matplotlib not available — skipping plot)")

## §2 Roofline Model for LLM Inference

The **roofline model** shows whether a workload is memory-bandwidth-bound
or compute-bound, based on its **arithmetic intensity** $I$ (FLOPs per byte).

$$\text{Perf}(I) = \min\bigl(\text{Peak FLOP/s},\; I \times \text{Peak BW}\bigr)$$

For decode with batch $B$ and params $P$ at $b$ bytes/param:

$$I_{\text{decode}}(B) = \frac{2PB}{Pb} = \frac{2B}{b}$$

In [ ]:
# === §2: Roofline Model ===

# --- 2a: Roofline analysis for different precisions and batch sizes ---

# Arithmetic intensity at decode
precisions = {
    "FP32": 4,
    "BF16": 2,
    "INT8": 1,
    "INT4": 0.5
}

# Ridge point: where BW-bound meets compute-bound
ridge_point = PEAK_FLOPS / HBM_BW  # FLOP/byte

print("=" * 65)
print("ROOFLINE ANALYSIS — H100 SXM")
print("=" * 65)
print(f"\nPeak FLOP/s (BF16): {fmt_si(PEAK_FLOPS, 'FLOP/s')}")
print(f"Peak HBM BW:        {fmt_si(HBM_BW, 'B/s')}")
print(f"Ridge point:        {ridge_point:.1f} FLOP/byte")
print()

# Arithmetic intensity table
print("Arithmetic intensity I = 2B/b at various batch sizes:")
print()
headers = ["Precision", "b (bytes)", "B=1", "B=8", "B=32", "B=128", "B=512"]
rows = []
for name, b in precisions.items():
    intensities = [2 * B / b for B in [1, 8, 32, 128, 512]]
    regime = ["BW" if I < ridge_point else "Compute" for I in intensities]
    row = [name, str(b)]
    for i, I in enumerate(intensities):
        row.append(f"{I:.1f} ({regime[i]})")
    rows.append(row)
print_table(headers, rows, col_width=14)

print(f"\n► Ridge point = {ridge_point:.1f} FLOP/byte.")
print(f"  Any intensity below this → BW-bound.")
print(f"  Decode at B=1 is ALWAYS BW-bound, regardless of precision.")

In [ ]:
# --- 2b: Roofline plot ---
if HAS_MPL:
    fig, ax = plt.subplots(figsize=(10, 6))

    # Roofline
    I_range = np.logspace(-1, 4, 500)
    perf_bw = I_range * HBM_BW
    perf_compute = np.full_like(I_range, PEAK_FLOPS)
    perf = np.minimum(perf_bw, perf_compute)

    ax.loglog(I_range, perf / 1e12, 'k-', linewidth=2, label='Roofline')
    ax.axvline(ridge_point, color='green', linestyle=':', alpha=0.5)
    ax.text(ridge_point * 1.1, 1, f'Ridge = {ridge_point:.0f}', color='green', fontsize=9)

    # Mark decode points for different precisions at B=1
    colors = {'FP32': 'red', 'BF16': 'blue', 'INT8': 'orange', 'INT4': 'purple'}
    for name, b in precisions.items():
        for B_val, marker in [(1, 'o'), (32, 's'), (256, '^')]:
            I_val = 2 * B_val / b
            perf_val = min(PEAK_FLOPS, I_val * HBM_BW)
            label = f"{name} B={B_val}" if marker == 'o' else None
            ax.plot(I_val, perf_val / 1e12, marker=marker, color=colors[name],
                    markersize=8, label=label)

    ax.set_xlabel('Arithmetic Intensity (FLOP/byte)')
    ax.set_ylabel('Performance (TFLOP/s)')
    ax.set_title('Roofline Model — H100 SXM — Decode Phase')
    ax.legend(loc='lower right', fontsize=8)
    ax.grid(True, alpha=0.3)
    ax.set_xlim(0.1, 5000)

    plt.tight_layout()
    plt.savefig('roofline_model.png', dpi=100, bbox_inches='tight')
    plt.show()
    print("Plot saved: roofline_model.png")
else:
    print("(Matplotlib not available — skipping plot)")

## §3 M/M/1 and M/M/c Queuing Simulation

LLM serving is fundamentally a **queuing system**: requests arrive
stochastically and compete for limited GPU resources.

**M/M/1 queue** (single server):
- Mean wait: $W_q = \frac{\rho}{\mu(1 - \rho)}$
- Mean queue length: $L_q = \frac{\rho^2}{1 - \rho}$

**M/M/c queue** (c servers):
- Erlang-C formula gives $P_0$, then $W_q = \frac{P_0 (c\rho)^c}{c! \cdot c\mu(1-\rho)^2}$

Key takeaway: at $\rho > 0.8$, wait times **explode**.

In [ ]:
# === §3: M/M/1 and M/M/c Queuing ===

from math import factorial, exp

# --- 3a: M/M/1 analytical results ---
def mm1_metrics(lam, mu):
    """M/M/1 queue metrics."""
    rho = lam / mu
    if rho >= 1.0:
        return {"rho": rho, "Wq": float('inf'), "Lq": float('inf'),
                "W": float('inf'), "L": float('inf')}
    Wq = rho / (mu * (1 - rho))
    W  = 1 / (mu * (1 - rho))        # total time in system
    Lq = rho**2 / (1 - rho)
    L  = rho / (1 - rho)             # total in system
    return {"rho": rho, "Wq": Wq, "Lq": Lq, "W": W, "L": L}

# --- 3b: M/M/c analytical results ---
def mmc_metrics(lam, mu, c):
    """M/M/c queue metrics."""
    rho = lam / (c * mu)  # per-server utilisation
    if rho >= 1.0:
        return {"rho": rho, "Wq": float('inf'), "Lq": float('inf'),
                "W": float('inf'), "L": float('inf'), "P0": 0}
    a = lam / mu  # offered load
    # Compute P0
    sum_terms = sum(a**k / factorial(k) for k in range(c))
    last_term = a**c / (factorial(c) * (1 - rho))
    P0 = 1.0 / (sum_terms + last_term)
    # Erlang-C (prob of queuing)
    C = (a**c / factorial(c)) * (1 / (1 - rho)) * P0
    Wq = C / (c * mu * (1 - rho))
    W  = Wq + 1 / mu
    Lq = lam * Wq
    L  = lam * W
    return {"rho": rho, "Wq": Wq, "Lq": Lq, "W": W, "L": L, "P0": P0, "C": C}

print("=" * 65)
print("QUEUING THEORY FOR LLM SERVING")
print("=" * 65)

# Scenario: λ = 10 req/s, μ = 3 req/s per GPU
lam = 10.0   # arrival rate
mu = 3.0     # service rate per GPU

print(f"\nArrival rate λ = {lam} req/s")
print(f"Service rate μ = {mu} req/s (per GPU)")

# M/M/1 (single GPU — won't work here since ρ > 1)
m1 = mm1_metrics(lam, mu)
print(f"\nM/M/1: ρ = {m1['rho']:.2f} → {'UNSTABLE' if m1['rho'] >= 1 else 'Stable'}")

# M/M/c for different c values
print(f"\nM/M/c results:")
headers = ["GPUs (c)", "ρ", "Wq (s)", "Lq", "P(queue)"]
rows = []
for c in [3, 4, 5, 6, 8, 10]:
    m = mmc_metrics(lam, mu, c)
    status = "UNSTABLE" if m['rho'] >= 1 else ""
    if m['rho'] < 1:
        rows.append([str(c), f"{m['rho']:.3f}", f"{m['Wq']:.3f}",
                     f"{m['Lq']:.2f}", f"{m['C']:.3f}"])
    else:
        rows.append([str(c), f"{m['rho']:.3f}", "∞", "∞", "—"])
print_table(headers, rows, col_width=14)

# Find minimum c for stability and for ρ < 0.7
for target_rho, label in [(0.999, "stable (ρ<1)"), (0.7, "ρ<0.7")]:
    c_min = int(np.ceil(lam / (mu * target_rho)))
    print(f"\n► Minimum GPUs for {label}: c = {c_min}")

In [ ]:
# --- 3c: Wait time vs utilisation ---
if HAS_MPL:
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

    # M/M/1: Wq vs ρ
    rhos = np.linspace(0.01, 0.98, 200)
    Wq_mm1 = rhos / (mu * (1 - rhos))
    ax1.plot(rhos, Wq_mm1 * 1000, 'b-', linewidth=2)
    ax1.axvline(0.7, color='green', linestyle=':', label='ρ=0.7 (target)')
    ax1.axvline(0.8, color='orange', linestyle=':', label='ρ=0.8 (danger)')
    ax1.axvline(0.9, color='red', linestyle=':', label='ρ=0.9 (critical)')
    ax1.set_xlabel('Server Utilisation ρ')
    ax1.set_ylabel('Mean Wait Time Wq (ms)')
    ax1.set_title('M/M/1: Wait Time vs Utilisation')
    ax1.set_ylim(0, 5000)
    ax1.legend()
    ax1.grid(True, alpha=0.3)

    # M/M/c: Wq vs c for fixed λ, μ
    c_values = range(4, 16)
    wq_c = []
    for c in c_values:
        m = mmc_metrics(lam, mu, c)
        wq_c.append(m['Wq'] * 1000 if m['rho'] < 1 else None)

    ax2.bar(list(c_values), [w if w is not None else 0 for w in wq_c],
            color=['red' if w is None or w > 500 else 'orange' if w > 200 else 'green'
                   for w in wq_c])
    ax2.set_xlabel('Number of GPUs (c)')
    ax2.set_ylabel('Mean Wait Time Wq (ms)')
    ax2.set_title(f'M/M/c: Wait Time vs GPU Count (λ={lam}, μ={mu})')
    ax2.grid(True, alpha=0.3, axis='y')

    plt.tight_layout()
    plt.savefig('queuing_analysis.png', dpi=100, bbox_inches='tight')
    plt.show()
    print("Plot saved: queuing_analysis.png")
else:
    print("(Matplotlib not available — skipping plot)")

## §4 Continuous vs. Static Batching

**Static batching**: All requests padded to max length; batch runs until
the *slowest* request finishes. GPU slots wasted for completed requests.

**Continuous batching (ORCA)**: Each decode step is the scheduling unit.
When a request finishes, its slot is immediately available for a new request.

Throughput improvement:

$$\text{Speedup} \approx \frac{\mathbb{E}[s_{\max}]}{\mathbb{E}[s]}$$

In [ ]:
# === §4: Continuous vs Static Batching ===

# --- 4a: Simulate static and continuous batching ---
np.random.seed(42)

TPOT_MS = 10.0   # ms per token
BATCH_SIZE = 10
N_REQUESTS = 100  # total requests to process

# Generate output lengths (geometric distribution, mean=100, clipped to 500)
output_lengths = np.clip(np.random.geometric(p=0.01, size=N_REQUESTS), 10, 500)

# --- Static batching ---
static_time = 0
static_tokens = 0
for batch_start in range(0, N_REQUESTS, BATCH_SIZE):
    batch = output_lengths[batch_start:batch_start + BATCH_SIZE]
    max_len = batch.max()
    batch_time = max_len * TPOT_MS  # All wait for longest
    static_time += batch_time
    static_tokens += batch.sum()
    # Wasted compute: (max_len - actual_len) * TPOT for each request

# --- Continuous batching (simplified simulation) ---
# Simulate token-by-token with slot re-filling
continuous_time = 0
continuous_tokens = 0
active_slots = list(output_lengths[:BATCH_SIZE])  # remaining tokens per slot
next_request = BATCH_SIZE

while any(s > 0 for s in active_slots) or next_request < N_REQUESTS:
    continuous_time += TPOT_MS  # one decode step
    # Decrement all active slots
    for i in range(len(active_slots)):
        if active_slots[i] > 0:
            active_slots[i] -= 1
            continuous_tokens += 1
    # Fill freed slots with new requests
    for i in range(len(active_slots)):
        if active_slots[i] == 0 and next_request < N_REQUESTS:
            active_slots[i] = output_lengths[next_request]
            next_request += 1

speedup = static_time / continuous_time

print("=" * 65)
print("CONTINUOUS vs STATIC BATCHING SIMULATION")
print("=" * 65)
print(f"\nRequests:           {N_REQUESTS}")
print(f"Batch size:         {BATCH_SIZE}")
print(f"TPOT:               {TPOT_MS} ms")
print(f"Output len range:   [{output_lengths.min()}, {output_lengths.max()}]")
print(f"Mean output len:    {output_lengths.mean():.1f}")
print(f"\nStatic batching:")
print(f"  Total time:       {static_time/1000:.1f} s")
print(f"  Throughput:       {static_tokens / (static_time/1000):.0f} tok/s")
print(f"\nContinuous batching:")
print(f"  Total time:       {continuous_time/1000:.1f} s")
print(f"  Throughput:       {continuous_tokens / (continuous_time/1000):.0f} tok/s")
print(f"\n► Speedup: {speedup:.2f}× (continuous over static)")
print(f"  ~E[s_max]/E[s] = {np.max(output_lengths)/np.mean(output_lengths):.2f}")

In [ ]:
# --- 4b: Visualise slot utilisation ---
if HAS_MPL:
    fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(14, 6), sharex=False)

    # Static batching - show first 3 batches
    batch_lens = [output_lengths[i*BATCH_SIZE:(i+1)*BATCH_SIZE] for i in range(3)]
    for batch_idx, batch in enumerate(batch_lens):
        max_l = batch.max()
        for slot, length in enumerate(batch):
            offset = sum(bl.max() for bl in batch_lens[:batch_idx])
            ax1.barh(slot, length, left=offset, color='steelblue', edgecolor='white', height=0.8)
            ax1.barh(slot, max_l - length, left=offset + length, color='lightgray',
                     edgecolor='white', height=0.8, alpha=0.5)

    ax1.set_ylabel('Batch Slot')
    ax1.set_xlabel('Tokens')
    ax1.set_title('Static Batching — Grey = Wasted Compute')
    ax1.set_yticks(range(BATCH_SIZE))

    # Output length distribution
    ax2.hist(output_lengths, bins=30, color='steelblue', edgecolor='white', alpha=0.8)
    ax2.axvline(output_lengths.mean(), color='red', linestyle='--', label=f'Mean = {output_lengths.mean():.0f}')
    ax2.axvline(output_lengths.max(), color='orange', linestyle='--', label=f'Max = {output_lengths.max()}')
    ax2.set_xlabel('Output Length (tokens)')
    ax2.set_ylabel('Count')
    ax2.set_title('Output Length Distribution')
    ax2.legend()

    plt.tight_layout()
    plt.savefig('batching_comparison.png', dpi=100, bbox_inches='tight')
    plt.show()
    print("Plot saved: batching_comparison.png")
else:
    print("(Matplotlib not available — skipping plot)")

## §5 PagedAttention Memory Analysis

**Naive KV allocation** pre-allocates $s_{\max}$ tokens for every request.
**PagedAttention** allocates in fixed-size blocks, wasting at most
(block\_size − 1) tokens per request.

$$\text{Waste}_{\text{naive}} = \sum_i (s_{\max} - s_i) \times M_{\text{KV/tok}}$$
$$\text{Waste}_{\text{paged}} \leq B \times (\text{block\_size} - 1) \times M_{\text{KV/tok}}$$

In [ ]:
# === §5: PagedAttention Memory Waste ===

# --- 5a: Compare naive vs paged allocation ---
np.random.seed(42)

N_REQUESTS = 64
S_MAX = 4096  # max sequence length
BLOCK_SIZE = 16  # tokens per block

# Model: LLaMA-3 70B, BF16
n_layers = 80
n_kv_heads = 8   # GQA
d_head = 128
b_kv = 2  # BF16
kv_per_tok = n_layers * 2 * n_kv_heads * d_head * b_kv  # bytes

# Generate random sequence lengths
seq_lengths = np.random.randint(100, 2000, size=N_REQUESTS)

# Naive waste
naive_allocated = N_REQUESTS * S_MAX * kv_per_tok
naive_used = seq_lengths.sum() * kv_per_tok
naive_waste = naive_allocated - naive_used

# Paged allocation
blocks_per_request = np.ceil(seq_lengths / BLOCK_SIZE).astype(int)
paged_allocated = blocks_per_request.sum() * BLOCK_SIZE * kv_per_tok
paged_waste = paged_allocated - naive_used  # internal fragmentation

# Additional requests possible with saved memory
saved_memory = naive_waste - paged_waste
avg_kv_per_request = seq_lengths.mean() * kv_per_tok
additional_requests = saved_memory / avg_kv_per_request

print("=" * 65)
print("PAGEDATTENTION MEMORY ANALYSIS")
print("=" * 65)
print(f"\nModel: LLaMA-3 70B, BF16")
print(f"KV per token: {fmt_bytes(kv_per_tok)} (all layers)")
print(f"Requests: {N_REQUESTS}")
print(f"Seq length range: [{seq_lengths.min()}, {seq_lengths.max()}]")
print(f"Mean seq length: {seq_lengths.mean():.0f}")
print(f"Block size: {BLOCK_SIZE} tokens")
print(f"s_max: {S_MAX}")
print()

headers = ["Metric", "Naive", "Paged"]
rows = [
    ["Allocated", fmt_bytes(naive_allocated), fmt_bytes(paged_allocated)],
    ["Used", fmt_bytes(naive_used), fmt_bytes(naive_used)],
    ["Wasted", fmt_bytes(naive_waste), fmt_bytes(paged_waste)],
    ["Waste %", f"{naive_waste/naive_allocated*100:.1f}%",
     f"{paged_waste/paged_allocated*100:.1f}%"],
]
print_table(headers, rows, col_width=20)

print(f"\n► Memory saved: {fmt_bytes(saved_memory)}")
print(f"► Additional requests possible: {additional_requests:.0f}")
print(f"► Paged wastes {naive_waste/paged_waste:.0f}× less than naive.")

In [ ]:
# --- 5b: Memory waste visualisation ---
if HAS_MPL:
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

    # Bar comparison
    categories = ['Naive', 'PagedAttention']
    used_gb = [naive_used / 1e9, naive_used / 1e9]
    wasted_gb = [naive_waste / 1e9, paged_waste / 1e9]

    ax1.bar(categories, used_gb, label='Used', color='steelblue')
    ax1.bar(categories, wasted_gb, bottom=used_gb, label='Wasted', color='lightcoral')
    ax1.set_ylabel('Memory (GB)')
    ax1.set_title('KV Cache Memory: Naive vs Paged')
    ax1.legend()
    ax1.grid(True, alpha=0.3, axis='y')

    # Waste vs block size
    block_sizes = [1, 2, 4, 8, 16, 32, 64, 128]
    wastes = []
    for bs in block_sizes:
        blocks = np.ceil(seq_lengths / bs).astype(int)
        alloc = blocks.sum() * bs * kv_per_tok
        w = alloc - naive_used
        wastes.append(w / 1e9)

    ax2.plot(block_sizes, wastes, 'ro-', linewidth=2)
    ax2.axhline(naive_waste/1e9, color='gray', linestyle='--', label='Naive waste')
    ax2.set_xlabel('Block Size (tokens)')
    ax2.set_ylabel('Internal Fragmentation (GB)')
    ax2.set_title('Paged Waste vs Block Size')
    ax2.set_xscale('log', base=2)
    ax2.legend()
    ax2.grid(True, alpha=0.3)

    plt.tight_layout()
    plt.savefig('pagedattention_analysis.png', dpi=100, bbox_inches='tight')
    plt.show()
    print("Plot saved: pagedattention_analysis.png")
else:
    print("(Matplotlib not available — skipping plot)")

## §6 KV Cache Budget Calculator

Given a model architecture and GPU configuration, compute:
1. KV bytes per token per layer
2. Total KV bytes per token
3. Available KV memory after model weights
4. Maximum concurrent tokens
5. Maximum concurrent requests at various context lengths

In [ ]:
# === §6: KV Cache Budget Calculator ===

def kv_budget(model_name, params_B, n_layers, n_kv_heads, d_head,
              bytes_per_param, bytes_per_kv, gpu_mem_GB, tp_degree):
    """Calculate KV cache budget for a given model/GPU config."""
    # KV per token per layer (K + V)
    kv_per_tok_per_layer = 2 * n_kv_heads * d_head * bytes_per_kv
    # Total KV per token (all layers)
    kv_per_tok = n_layers * kv_per_tok_per_layer
    # KV per token per GPU (TP splits KV heads)
    kv_per_tok_per_gpu = kv_per_tok / tp_degree
    # Model memory per GPU
    model_mem = params_B * 1e9 * bytes_per_param / tp_degree
    # Available for KV (account for ~2GB overhead)
    avail = gpu_mem_GB * 1e9 - model_mem - 2e9
    # Max concurrent tokens
    max_tokens = avail / kv_per_tok_per_gpu if avail > 0 else 0

    return {
        "model": model_name,
        "kv_per_tok_per_layer": kv_per_tok_per_layer,
        "kv_per_tok": kv_per_tok,
        "kv_per_tok_per_gpu": kv_per_tok_per_gpu,
        "model_mem_per_gpu": model_mem,
        "avail_kv_mem": avail,
        "max_tokens": max_tokens,
    }

# Define model configs
configs = [
    ("LLaMA-3 8B",  8,  32, 8,  128, 2, 2, 80, 1),
    ("LLaMA-3 70B", 70, 80, 8,  128, 2, 2, 80, 2),
    ("LLaMA-3 70B", 70, 80, 8,  128, 2, 2, 80, 4),
    ("LLaMA-3 405B",405,126,8,  128, 2, 2, 80, 8),
    ("LLaMA-3 70B (INT4)", 70, 80, 8, 128, 0.5, 2, 80, 1),
]

print("=" * 80)
print("KV CACHE BUDGET CALCULATOR")
print("=" * 80)

for args in configs:
    r = kv_budget(*args)
    print(f"\n{'─' * 60}")
    print(f"  {r['model']} | TP={args[8]} | GPU={args[7]}GB | Precision={args[5]}B model, {args[6]}B KV")
    print(f"{'─' * 60}")
    print(f"  KV/token/layer:   {fmt_bytes(r['kv_per_tok_per_layer'])}")
    print(f"  KV/token (total): {fmt_bytes(r['kv_per_tok'])}")
    print(f"  KV/token/GPU:     {fmt_bytes(r['kv_per_tok_per_gpu'])}")
    print(f"  Model mem/GPU:    {fmt_bytes(r['model_mem_per_gpu'])}")
    print(f"  Available KV mem: {fmt_bytes(r['avail_kv_mem'])}")
    print(f"  Max tokens:       {r['max_tokens']:,.0f}")
    for ctx in [512, 1024, 2048, 4096, 8192]:
        max_req = r['max_tokens'] / ctx
        if max_req >= 1:
            print(f"    @ {ctx:>5} ctx:   {max_req:.0f} concurrent requests")
        else:
            print(f"    @ {ctx:>5} ctx:   Cannot fit even 1 request")

## §7 Speculative Decoding Throughput Model

**Speculative decoding** uses a cheap draft model to propose $K$ tokens,
then the target model verifies them in one pass.

Expected accepted tokens (constant $\alpha$):

$$\mathbb{E}[\text{accepted}] = \frac{1 - \alpha^{K+1}}{1 - \alpha} - 1$$

Speed-up:

$$S = \frac{(\mathbb{E}[\text{accepted}] + 1) \times t_{\text{AR}}}{K \cdot t_d + t_v}$$

In [ ]:
# === §7: Speculative Decoding ===

# --- 7a: Expected tokens and speedup ---
def spec_decode_analysis(alpha, K, gamma=0.1):
    """
    alpha: mean acceptance rate
    K: draft length
    gamma: draft-to-target cost ratio (t_d / t_AR)
    """
    # Expected accepted tokens
    E_accepted = (1 - alpha**(K+1)) / (1 - alpha) - 1
    # Total tokens per cycle (accepted + 1 correction)
    n_cycle = E_accepted + 1
    # Time per cycle (normalised by t_AR)
    t_cycle = K * gamma + 1  # K*t_d + t_v, in units of t_AR
    # Speedup
    speedup = n_cycle / t_cycle
    return E_accepted, n_cycle, t_cycle, speedup

print("=" * 70)
print("SPECULATIVE DECODING ANALYSIS")
print("=" * 70)

# Vary alpha for fixed K=4
K_fixed = 4
gamma = 0.1  # draft is 10x faster
print(f"\nDraft length K = {K_fixed}, draft cost ratio γ = {gamma}")
print()

headers = ["α", "E[accepted]", "Tokens/cycle", "Speedup"]
rows = []
for alpha in [0.5, 0.6, 0.7, 0.75, 0.8, 0.85, 0.9, 0.95]:
    E, n, t, S = spec_decode_analysis(alpha, K_fixed, gamma)
    rows.append([f"{alpha:.2f}", f"{E:.2f}", f"{n:.2f}", f"{S:.2f}×"])
print_table(headers, rows, col_width=14)

# Vary K for fixed alpha=0.75
alpha_fixed = 0.75
print(f"\nAcceptance rate α = {alpha_fixed}, draft cost ratio γ = {gamma}")
print()

headers = ["K", "E[accepted]", "Tokens/cycle", "Speedup"]
rows = []
for K in [1, 2, 3, 4, 5, 6, 8, 10]:
    E, n, t, S = spec_decode_analysis(alpha_fixed, K, gamma)
    rows.append([str(K), f"{E:.2f}", f"{n:.2f}", f"{S:.2f}×"])
print_table(headers, rows, col_width=14)

# Break-even analysis: find alpha where S = 1
# S = 1 when n_cycle = t_cycle → (1 - α^(K+1))/(1-α) = K*γ + 1
from scipy.optimize import brentq
try:
    def breakeven_eq(a, K, g):
        return (1 - a**(K+1)) / (1 - a) - (K * g + 1)
    a_be = brentq(breakeven_eq, 0.01, 0.99, args=(K_fixed, gamma))
    print(f"\n► Break-even α (S=1) at K={K_fixed}, γ={gamma}: α = {a_be:.4f}")
    print(f"  Below this acceptance rate, speculation is SLOWER than standard decode.")
except Exception:
    # Manual binary search
    lo, hi = 0.01, 0.99
    for _ in range(50):
        mid = (lo + hi) / 2
        _, _, _, S = spec_decode_analysis(mid, K_fixed, gamma)
        if S > 1: hi = mid
        else: lo = mid
    print(f"\n► Break-even α (S=1) at K={K_fixed}, γ={gamma}: α ≈ {mid:.4f}")

In [ ]:
# --- 7b: Speedup heatmap ---
if HAS_MPL:
    alphas = np.linspace(0.3, 0.95, 30)
    Ks = np.arange(1, 11)
    speedup_grid = np.zeros((len(Ks), len(alphas)))

    for i, K in enumerate(Ks):
        for j, a in enumerate(alphas):
            _, _, _, S = spec_decode_analysis(a, K, gamma)
            speedup_grid[i, j] = S

    fig, ax = plt.subplots(figsize=(10, 6))
    im = ax.imshow(speedup_grid, aspect='auto', origin='lower',
                   extent=[alphas[0], alphas[-1], Ks[0]-0.5, Ks[-1]+0.5],
                   cmap='RdYlGn', vmin=0.5, vmax=4.0)
    ax.set_xlabel('Acceptance Rate α')
    ax.set_ylabel('Draft Length K')
    ax.set_title(f'Speculative Decoding Speedup (γ={gamma})')
    cbar = plt.colorbar(im, ax=ax, label='Speedup ×')

    # Contour at S=1 (break-even)
    ax.contour(alphas, Ks, speedup_grid, levels=[1.0], colors='black',
               linewidths=2, linestyles='--')

    plt.tight_layout()
    plt.savefig('speculative_decoding.png', dpi=100, bbox_inches='tight')
    plt.show()
    print("Plot saved: speculative_decoding.png")
else:
    print("(Matplotlib not available — skipping plot)")

## §8 Cost Per Million Tokens

$$\text{CPM} = \frac{\text{GPU cost/hr} \times 10^6}{\text{throughput (tok/s)} \times 3600}$$

This section computes CPM across different model sizes,
precisions, and hardware configurations.

In [ ]:
# === §8: Cost Per Million Tokens ===

# --- 8a: CPM calculator ---
def compute_cpm(gpu_cost_hr, n_gpus, throughput_toks):
    """Compute cost per million output tokens."""
    total_cost_hr = gpu_cost_hr * n_gpus
    tokens_per_hour = throughput_toks * 3600
    cpm = total_cost_hr * 1e6 / tokens_per_hour
    return cpm

# Scenarios
scenarios = [
    # (label, gpu_cost, n_gpus, tok/s)
    ("LLaMA-3 8B, BF16, 1×H100",   3.00, 1, 8000),
    ("LLaMA-3 8B, INT4, 1×H100",    3.00, 1, 15000),
    ("LLaMA-3 8B, BF16, 1×A100",    2.00, 1, 4500),
    ("LLaMA-3 70B, BF16, 2×H100",   3.00, 2, 2500),
    ("LLaMA-3 70B, INT4, 2×H100",   3.00, 2, 5500),
    ("LLaMA-3 70B, INT4, 4×A100",   2.00, 4, 3000),
    ("LLaMA-3 405B, BF16, 8×H100",  3.00, 8, 1200),
    ("LLaMA-3 405B, INT4, 8×H100",  3.00, 8, 2800),
]

print("=" * 80)
print("COST PER MILLION TOKENS (CPM) — OUTPUT TOKENS")
print("=" * 80)
print()

headers = ["Configuration", "GPUs", "$/hr", "Tok/s", "CPM ($)"]
rows = []
for label, cost, n, tps in scenarios:
    cpm = compute_cpm(cost, n, tps)
    rows.append([label, str(n), f"${cost*n:.2f}",
                 f"{tps:,}", f"${cpm:.3f}"])
print_table(headers, rows, col_width=30)

# Daily/monthly cost for 1B tokens/day
print("\n" + "─" * 60)
print("DAILY COST FOR 1 BILLION OUTPUT TOKENS")
print("─" * 60)
for label, cost, n, tps in scenarios:
    cpm = compute_cpm(cost, n, tps)
    daily = cpm * 1e9 / 1e6  # cost for 1B tokens
    monthly = daily * 30
    print(f"  {label:40s}: ${daily:>8,.0f}/day  ${monthly:>10,.0f}/month")

## §9 Throughput-Latency Pareto Curve

As batch size increases, throughput improves but latency eventually
degrades. The **Pareto-optimal** operating point maximises throughput
subject to an SLO constraint on TPOT.

In [ ]:
# === §9: Throughput-Latency Pareto Curve ===

# --- 9a: Model the throughput-latency tradeoff ---
# BW-bound regime: throughput ∝ B, TPOT constant
# Compute-bound regime: throughput saturates, TPOT ∝ B
# Transition at B*

def throughput_latency(B, P, b, BW, peak_flops):
    """Compute throughput and TPOT for batch size B."""
    # Time per decode step
    t_bw = P * b / BW              # BW-bound time
    t_compute = 2 * P * B / peak_flops  # compute-bound time
    t_step = max(t_bw, t_compute)  # actual time per step

    throughput = B / t_step         # tokens/s
    tpot = t_step / B * 1000       # ms per user per token
    return throughput, tpot

batch_range = np.arange(1, 513)

# Compute for different model sizes
models = [
    ("8B BF16",  8e9,  2),
    ("70B BF16", 70e9, 2),
    ("8B INT4",  8e9,  0.5),
]

print("=" * 70)
print("THROUGHPUT-LATENCY PARETO ANALYSIS")
print("=" * 70)

for name, P, b in models:
    B_star = PEAK_FLOPS * b / (2 * HBM_BW)
    print(f"\n{name}: B* = {B_star:.0f}")

    # Find SLO-optimal batch size for different TPOT constraints
    for slo_tpot in [10, 20, 50, 100]:
        best_B = 1
        for B in batch_range:
            _, tpot = throughput_latency(B, P, b, HBM_BW, PEAK_FLOPS)
            if tpot <= slo_tpot:
                best_B = B
        thr, tpot = throughput_latency(best_B, P, b, HBM_BW, PEAK_FLOPS)
        print(f"  SLO TPOT ≤ {slo_tpot:>3}ms: B* = {best_B:>4}, "
              f"throughput = {thr:>10,.0f} tok/s, TPOT = {tpot:.1f}ms")

In [ ]:
# --- 9b: Pareto plot ---
if HAS_MPL:
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

    for name, P, b in models:
        thrs, tpots = [], []
        for B in batch_range:
            thr, tp = throughput_latency(B, P, b, HBM_BW, PEAK_FLOPS)
            thrs.append(thr)
            tpots.append(tp)

        ax1.plot(batch_range, np.array(thrs)/1000, label=name, linewidth=2)
        ax2.plot(batch_range, tpots, label=name, linewidth=2)

    ax1.set_xlabel('Batch Size')
    ax1.set_ylabel('Throughput (K tok/s)')
    ax1.set_title('Throughput vs Batch Size')
    ax1.legend()
    ax1.grid(True, alpha=0.3)

    ax2.set_xlabel('Batch Size')
    ax2.set_ylabel('TPOT (ms)')
    ax2.set_title('Per-User Latency vs Batch Size')
    ax2.axhline(30, color='red', linestyle='--', alpha=0.5, label='SLO: 30ms')
    ax2.legend()
    ax2.grid(True, alpha=0.3)
    ax2.set_ylim(0, 150)

    plt.tight_layout()
    plt.savefig('pareto_curve.png', dpi=100, bbox_inches='tight')
    plt.show()
    print("Plot saved: pareto_curve.png")
else:
    print("(Matplotlib not available — skipping plot)")

## §10 Capacity Planning and Fleet Sizing

Given expected traffic (requests/s), SLOs, and per-GPU performance,
determine the minimum GPU count for stable operation.

$$N_{\text{GPU,min}} = \left\lceil \frac{\lambda \times n_{\text{output}}}{\Theta_{\text{GPU}} \times (1 - \text{headroom})} \right\rceil$$

In [ ]:
# === §10: Capacity Planning ===

# --- 10a: Fleet sizing calculator ---
def fleet_sizing(lam_rps, avg_output_tokens, gpu_throughput,
                 headroom=0.3, burst_factor=2.0, tp_degree=1):
    """
    lam_rps: mean request rate (req/s)
    avg_output_tokens: average output length
    gpu_throughput: tokens/s per GPU
    headroom: safety margin (e.g., 0.3 = 30%)
    burst_factor: ratio of P99 to mean arrival rate
    tp_degree: GPUs per model instance for TP
    """
    # Tokens/s demand
    demand_mean = lam_rps * avg_output_tokens
    demand_burst = demand_mean * burst_factor

    # Supply per GPU instance (TP group)
    supply_per_instance = gpu_throughput

    # Instances needed
    instances_mean = np.ceil(demand_mean / (supply_per_instance * (1 - headroom)))
    instances_burst = np.ceil(demand_burst / (supply_per_instance * (1 - headroom)))

    # Total GPUs
    gpus_mean = int(instances_mean * tp_degree)
    gpus_burst = int(instances_burst * tp_degree)

    # Utilisation at mean load
    util_mean = demand_mean / (instances_mean * supply_per_instance)

    return {
        "demand_mean_tps": demand_mean,
        "demand_burst_tps": demand_burst,
        "instances_mean": int(instances_mean),
        "instances_burst": int(instances_burst),
        "gpus_mean": gpus_mean,
        "gpus_burst": gpus_burst,
        "utilisation_mean": util_mean
    }

print("=" * 75)
print("CAPACITY PLANNING — FLEET SIZING")
print("=" * 75)

# Scenarios
scenarios = [
    ("Small startup",    5,   200, 8000,  0.3, 1.5, 1),
    ("Mid-size app",    50,   200, 5000,  0.3, 2.0, 2),
    ("Large platform", 500,   200, 5000,  0.3, 2.0, 2),
    ("Mega-scale",    5000,   200, 2500,  0.3, 2.5, 4),
]

for label, lam, avg_out, thr, head, burst, tp in scenarios:
    r = fleet_sizing(lam, avg_out, thr, head, burst, tp)
    print(f"\n{'─' * 55}")
    print(f"  {label} (λ={lam} req/s, avg_out={avg_out}, TP={tp})")
    print(f"{'─' * 55}")
    print(f"  Token demand (mean): {r['demand_mean_tps']:,.0f} tok/s")
    print(f"  Token demand (burst): {r['demand_burst_tps']:,.0f} tok/s")
    print(f"  GPUs needed (mean): {r['gpus_mean']}")
    print(f"  GPUs needed (burst): {r['gpus_burst']}")
    print(f"  Mean utilisation: {r['utilisation_mean']:.1%}")

In [ ]:
# --- 10b: Cost projection over traffic levels ---
if HAS_MPL:
    lam_range = np.logspace(0, 4, 100)  # 1 to 10000 req/s
    gpu_cost_hr = 3.00

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

    for label, thr, tp, color in [
        ("8B BF16 TP=1",  8000, 1, 'blue'),
        ("70B BF16 TP=2", 2500, 2, 'red'),
        ("70B INT4 TP=2", 5500, 2, 'green'),
    ]:
        gpus = [fleet_sizing(l, 200, thr, 0.3, 1.0, tp)['gpus_mean'] for l in lam_range]
        monthly_cost = np.array(gpus) * gpu_cost_hr * 24 * 30

        ax1.loglog(lam_range, gpus, label=label, color=color, linewidth=2)
        ax2.loglog(lam_range, monthly_cost / 1000, label=label, color=color, linewidth=2)

    ax1.set_xlabel('Request Rate (req/s)')
    ax1.set_ylabel('GPUs Required')
    ax1.set_title('GPU Count vs Traffic')
    ax1.legend()
    ax1.grid(True, alpha=0.3)

    ax2.set_xlabel('Request Rate (req/s)')
    ax2.set_ylabel('Monthly Cost ($K)')
    ax2.set_title('Monthly GPU Cost vs Traffic')
    ax2.legend()
    ax2.grid(True, alpha=0.3)

    plt.tight_layout()
    plt.savefig('capacity_planning.png', dpi=100, bbox_inches='tight')
    plt.show()
    print("Plot saved: capacity_planning.png")
else:
    print("(Matplotlib not available — skipping plot)")

In [ ]:
# === THEORY NOTEBOOK COMPLETE ===
print("="*70)
print("  CHAPTER 13 — SERVING AND SYSTEMS TRADEOFFS: THEORY COMPLETE")
print("="*70)
print()
print("Key takeaways:")
print()
print("  §1  Bandwidth-bound decode: tok/s = B × BW / (P × b)")
print("  §2  Roofline model: decode is BW-bound until B > B* ≈ 295")
print("  §3  Queuing theory: Wq → ∞ as ρ → 1; provision for ρ < 0.7")
print("  §4  Continuous batching: 2–5× throughput over static")
print("  §5  PagedAttention: ~200× less memory waste than naive")
print("  §6  KV budget: 320 KB/token for 70B BF16; limits concurrency")
print("  §7  Speculative decoding: 2–3× speedup at α=0.75, K=4")
print("  §8  CPM: $0.10–$5.56 depending on model and hardware")
print("  §9  Pareto curve: maximise throughput subject to TPOT SLO")
print("  §10 Capacity planning: N = ⌈λ × n_out / (Θ × (1-headroom))⌉")
print()
print("  These tools let you answer: How many GPUs do I need,")
print("  and how much will it cost?")